In [ ]:
import jax
import jax.numpy as jnp
import optax
import diffrax

from dmpe.utils.signals import aprbs
import exciting_environments as excenvs
from dmpe.models.models import NeuralEulerODEPMSM
from dmpe.algorithms import excite_with_dmpe
from dmpe.utils.density_estimation import get_target_distribution
import dmpe.utils.env_utils.pmsm_utils as pmsm_utils
from dmpe.evaluation.callbacks import plot_sequence_callback

In [ ]:
def setup_env(rpm):
    env = pmsm_utils.ExcitingPMSM(
        initial_rpm=rpm,
        batch_size=1,
        saturated=True,
        LUT_motor_name="BRUSA",
        static_params={
            "p": 3,
            "r_s": 17.932e-3,
            "l_d": jnp.nan,
            "l_q": jnp.nan,
            "psi_p": 65.65e-3,
            "deadtime": 0,
        },
        solver=diffrax.Tsit5(),
    )
    penalty_function = lambda observations, actions: pmsm_utils.PMSM_penalty(env, observations, actions)
    
    return env, penalty_function


def setup_exp(env, penalty_function, consider_action_distribution):
    alg_params = dict(
        bandwidth=0.08,
        n_prediction_steps=5,
        points_per_dim=21,
        grid_extend=1.05,
        excitation_optimizer=optax.adabelief(1e-2),
        n_opt_steps=50,
        start_optimizing=5,
        consider_action_distribution=consider_action_distribution,
        penalty_function=penalty_function,
        target_distribution=None,
        clip_action=False,
        n_starts=10,
        reuse_proposed_actions=True,
    )
    
    alg_params["target_distribution"] = get_target_distribution(
        points_per_dim=alg_params["points_per_dim"],
        bandwidth=alg_params["bandwidth"],
        grid_extend=alg_params["grid_extend"],
        consider_action_distribution=consider_action_distribution,
        penalty_function=penalty_function,
    )
    
    # parameters for the training of the model
    model_trainer_params = dict(
        start_learning=alg_params["n_prediction_steps"],
        training_batch_size=64,
        n_train_steps=5,
        sequence_length=alg_params["n_prediction_steps"],
        featurize=lambda x: x,
        model_lr=1e-4,
    )
    
    # parameters of the model itself
    model_params = dict(obs_dim=2, action_dim=env.action_dim, width_size=64, depth=3, key=None)
    
    
    # setup the whole experiment parameter dict
    exp_params = dict(
        seed=int(42),
        n_time_steps=1500,
        model_class=NeuralEulerODEPMSM,
        env_params=None,
        alg_params=alg_params,
        model_trainer_params=model_trainer_params,
        model_params=model_params,
    )

    return exp_params

# Apply DMPE to the permanent magnet synchronous motor (PMSM):

## do not consider action distribution:

In [ ]:
env, penalty_function = setup_env(rpm=3000)
consider_action_distribution = False

exp_params = setup_exp(env, penalty_function, consider_action_distribution)

In [ ]:
# setup PRNG
key = jax.random.PRNGKey(seed=exp_params["seed"])
data_key, model_key, loader_key, expl_key, key = jax.random.split(key, 5)
exp_params["model_params"]["key"] = model_key

data_keys = jax.random.split(data_key, env.action_dim)

# initial guess for U_k
proposed_actions = (
    jnp.hstack(
        [
            aprbs(exp_params["alg_params"]["n_prediction_steps"], env.batch_size, 1, 10, data_keys[j])[0]
            for j in range(env.action_dim)
        ]
    )
    / 5
)

# run the algorithm
observations, actions, model, density_estimate, losses, proposed_actions, callback_out = excite_with_dmpe(
    env, exp_params, proposed_actions, loader_key, expl_key, callback=plot_sequence_callback, callback_every=500
)

In [ ]:
from dmpe.evaluation.plotting_utils import plot_sequence, plot_feature_combinations
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}"

fig = plot_sequence(observations, actions, env.tau, env.obs_description, env.action_description)
plt.savefig("../fig/MIMO_example_trajectories_no_action_coverage.png")
plt.show()

fig = plot_feature_combinations(
    data=jnp.concatenate([observations[:-1], actions], axis=-1),
    labels=["$\\tilde{i}_\mathrm{d}$", "$\\tilde{i}_\mathrm{q}$", "$\\tilde{v}_\mathrm{d}$", "$\\tilde{v}_\mathrm{q}$"],
    mode="contourf",
    bandwidth=exp_params["alg_params"]["bandwidth"],
    points_per_dim=exp_params["alg_params"]["points_per_dim"],
)
plt.savefig("../fig/MIMO_example_coverage_no_action_coverage.png")
plt.show()

## do consider action distribution:

In [ ]:
env, penalty_function = setup_env(rpm=3000)
consider_action_distribution = True

exp_params = setup_exp(env, penalty_function, consider_action_distribution)

In [ ]:
# setup PRNG
key = jax.random.PRNGKey(seed=exp_params["seed"])
data_key, model_key, loader_key, expl_key, key = jax.random.split(key, 5)
exp_params["model_params"]["key"] = model_key

data_keys = jax.random.split(data_key, env.action_dim)

# initial guess for U_k
proposed_actions = (
    jnp.hstack(
        [
            aprbs(exp_params["alg_params"]["n_prediction_steps"], env.batch_size, 1, 10, data_keys[j])[0]
            for j in range(env.action_dim)
        ]
    )
    / 5
)

# run the algorithm
observations, actions, model, density_estimate, losses, proposed_actions, callback_out = excite_with_dmpe(
    env, exp_params, proposed_actions, loader_key, expl_key, callback=plot_sequence_callback, callback_every=500
)

In [ ]:
from dmpe.evaluation.plotting_utils import plot_sequence, plot_feature_combinations
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}"

fig = plot_sequence(observations, actions, env.tau, env.obs_description, env.action_description)
plt.show()

fig = plot_feature_combinations(
    data=jnp.concatenate([observations[:-1], actions], axis=-1),
    labels=["$\\tilde{i}_\mathrm{d}$", "$\\tilde{i}_\mathrm{q}$", "$\\tilde{v}_\mathrm{d}$", "$\\tilde{v}_\mathrm{q}$"],
    mode="contourf",
    bandwidth=exp_params["alg_params"]["bandwidth"],
    points_per_dim=exp_params["alg_params"]["points_per_dim"],
)
plt.savefig("../fig/MIMO_example_coverage_action_coverage.png")
plt.show()